# RF2_ab vs vanilla RF2 — guided walk-through

A piecewise tutorial answering three questions about how RFantibody's fine-tuned RF2_ab differs from upstream `uw-ipd/RoseTTAFold2`:

1. **Case 1** — what do RF2_ab's extra weights actually encode?
2. **Case 2** — is the preprocessing pipeline also antibody-specific, or just the weights?
3. **Case 3** — empirically, how much worse is vanilla RF2 on an antibody-antigen complex?

Runnable top-to-bottom on an RTX 4080 (16 GB). Total wall-clock: ~5 min on a cold run, dominated by 3 ColabFold MSA fetches. All cells are idempotent — re-running won't re-do expensive steps.

Companion narrative doc: `../rf2-vs-rfab-tutorial.md`.

## Prerequisites

Before running this notebook, run from the repo root:

```bash
make install-rfantibody      # uv sync + RFantibody weights
make install-rf2-vanilla     # clone uw-ipd/RoseTTAFold2 + RF2_jan24.pt
```

The next cell verifies that both installs are in place and discovers the paths. Edit the env vars at the top if yours differ.

In [ ]:
import os
import subprocess
from pathlib import Path

# Adjust these if your layout differs
RFA_ROOT = Path(os.environ.get(
    "RFA_ROOT",
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
))
RF2_VANILLA_DIR = Path(os.environ.get(
    "RF2_VANILLA_DIR",
    Path.home() / "rf2-vanilla" / "RoseTTAFold2",
))

required = {
    "RF2_ab weights":      RFA_ROOT / "weights" / "RF2_ab.pt",
    "RFantibody src":      RFA_ROOT / "src" / "rfantibody" / "rf2",
    "diff script":         RFA_ROOT / "scripts" / "diff_rf2_checkpoints.py",
    "compare script":      RFA_ROOT / "scripts" / "compare_rf2_vs_rfab.py",
    "vanilla RF2 repo":    RF2_VANILLA_DIR / "network" / "predict.py",
    "vanilla RF2 weights": RF2_VANILLA_DIR / "network" / "weights" / "RF2_jan24.pt",
    "test input PDB":      RFA_ROOT / "test" / "rf2" / "inputs_for_test" / "ab_proteinmpnn_output.pdb",
}
for label, path in required.items():
    ok = "OK " if path.exists() else "MISSING"
    print(f"  [{ok}] {label:25s} -> {path}")

missing = [p for p in required.values() if not p.exists()]
assert not missing, f"Install prereqs first. Missing: {missing}"
print("\nAll prerequisites present.")

---

## Case 1 — what do the extra RF2_ab weights encode?

**Claim to verify:** RF2_ab's state-dict has the same topology as vanilla RF2 except for three template-embedding layers whose input widths are wider by +2 / +1 / +1. Is that "four extra Ab-specific input features," or one feature counted four times?

### Step 1.1 — state-dict diff

Load both checkpoints and report key differences + tensor-shape mismatches.

In [ ]:
result = subprocess.run(
    ["uv", "run", "python", "scripts/diff_rf2_checkpoints.py",
     "weights/RF2_ab.pt", "weights/RF2_jan24.pt"],
    cwd=str(RFA_ROOT), capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Two independent incompatibilities show up:

- **Template embeddings (+2 / +1 / +1)** — the topic of this case.
- **Different `bind_pred` head** — RF2_ab has a v2 `BinderNetwork` (`bind_pred.downsample` + `bind_pred.rbf2attn`, enabled by `new_pbind: True`), vanilla has a v1 `bind_pred.classify`. Independent reason for non-swappability; we won't analyse it further here.

Now let's trace *where* the +2/+1/+1 comes from.

### Step 1.2 — trace `d_t1d` through the code

All three template shape differences should trace back to a single config knob, `d_t1d` (template 1D feature width).

In [ ]:
def show(path: str, pattern: str, label: str, limit: int = 15):
    """Print grep hits from a file relative to RFA_ROOT."""
    full = RFA_ROOT / path
    out = subprocess.run(
        ["grep", "-nE", pattern, str(full)],
        capture_output=True, text=True,
    ).stdout.splitlines()[:limit]
    print(f"--- {label}: {path} ---")
    for line in out:
        print(line)
    print()

show("src/rfantibody/rf2/config/base.yaml",
     r"d_t1d", "config override")
show("src/rfantibody/rf2/network/Embeddings.py",
     r"d_t1d|self\.emb = nn\.Linear|self\.proj_t1d = nn\.Linear",
     "layer instantiations", limit=20)

Reading the three layer formulas:

| layer | formula | vanilla (d_t1d=22) | RFa (d_t1d=23) | Δ |
|---|---|---:|---:|---:|
| `templ_emb.emb` | `d_t1d*2 + d_t2d` (t1d appears once per side of a pair) | 88 | 90 | **+2** |
| `templ_emb.proj_t1d` | `d_t1d + d_tor` | 52 | 53 | **+1** |
| `templ_emb.templ_stack.proj_t1d` | `d_t1d` | 22 | 23 | **+1** |

So `d_t1d: 22 → 23` alone explains all four extra input-slot entries the state-dict diff reports. **One scalar feature, used four times** — not four independent features.

### Step 1.3 — what does the extra channel encode?

Look at where dim 22 of the t1d tensor is populated in the preprocessor:

In [ ]:
preprocess_py = RFA_ROOT / "src/rfantibody/rf2/modules/preprocess.py"
lines = preprocess_py.read_text().splitlines()
# make_t1d lives around line 174-197
for i, line in enumerate(lines[173:197], start=174):
    print(f"{i:4d}| {line}")

Dim 22 of t1d (line 195) is the **hotspot mask** — a binary per-residue flag set only on template 0, identifying target-surface residues that the antibody should bind to.

### Case 1 conclusion

RF2_ab is not "antibody-aware" because of H/L chain flags or CDR masks. It is **interface-aware**: exactly one extra per-residue scalar (a hotspot on the target) is wired into four weight-matrix input slots across three template-encoder layers. Swapping the `.pt` is impossible because the template encoder's input widths don't match, and (independently) because the binder head was rebuilt.

---

## Case 2 — is the pipeline also antibody-specific?

**Claim to verify:** Ben's assertion that the pipeline (not just the weights) has Ab-specific preprocessing. We look for four specific signatures and one missing one.

### 2.1 — `+200` chain offset (most load-bearing)

In `make_RF_idx`, antibody chains get a fixed 200-residue idx buffer after the target. This shapes RF2's pair-attention distance prior — without it, inter-chain attention behaves as if H/L/T were one long chain.

In [ ]:
for i, line in enumerate(lines[281:316], start=282):
    print(f"{i:4d}| {line}")

### 2.2 — fused H+L `same_chain` mask

Both antibody chains are treated as one chain for pair attention and loss clamps; only T is a separate chain.

In [ ]:
pose_util_py = RFA_ROOT / "src/rfantibody/rf2/modules/pose_util.py"
pose_lines = pose_util_py.read_text().splitlines()
for i, line in enumerate(pose_lines[159:170], start=160):
    print(f"{i:4d}| {line}")

### 2.3 — antibody sequence masking in t1d

The antibody AA one-hot is zeroed and confidence set to 0 for antibody residues; target keeps AA + confidence 1. Baked-in design-mode assumption: "antibody sequence is unknown / to-be-designed; target is known."

In [ ]:
for i, line in enumerate(lines[185:193], start=186):
    print(f"{i:4d}| {line}")

### 2.4 — no MSA handling at all

The one claim that doesn't hold up is "MSA handling differs between the two models" — because RFantibody's inference path doesn't do any MSA search. It fakes a 1-row MSA from the query sequence:

In [ ]:
for i, line in enumerate(lines[151:172], start=152):
    print(f"{i:4d}| {line}")

### Case 2 conclusion

Four load-bearing Ab-specific preprocessing choices:

1. **+200 chain offset** (`preprocess.py:282-316`) — single most important
2. **Fused H+L `same_chain`** (`pose_util.py:160-170`)
3. **Antibody sequence masking** (`preprocess.py:186-192`)
4. **Hotspot channel injection** (`preprocess.py:195`, from Case 1)

None are new algorithms — they're configuration/masking choices layered on top of vanilla RF2's existing multi-chain machinery. The heavier Ab specialization lives in the *weights themselves*, not in preprocessing transformations.

---

## Case 3 — how much worse is vanilla RF2 on an antibody-antigen complex?

Feed the *same* H/L/T sequences to both models through their *own* native pipelines:

- **RFantibody RF2**: `rf2` CLI → synthetic 1-seq MSA, +200 offset, hotspots, RF2_ab weights.
- **Vanilla RF2**: `predict.py -inputs H.a3m L.a3m T.a3m` → ColabFold-retrieved per-chain MSAs, no hotspots, RF2_jan24 weights.

Test input is RFantibody's own test case (`test/rf2/inputs_for_test/ab_proteinmpnn_output.pdb`), a 478-residue designed Ab-Ag complex (H=121, L=106, T=251).

**Expected timing on a 4080:**
- MSA fetch: ~3-4 min (network-bound, ColabFold queue)
- RFa RF2: ~30 s
- Vanilla RF2: ~20 s
- Comparison: <1 s

### 3.1 — Stage input and extract sequences

In [ ]:
import shutil

CASE = RFA_ROOT / "outputs" / "rf2_comparison" / "case1_ab_proteinmpnn"
for sub in ("inputs_rfa", "a3m_vanilla", "rfa_out", "vanilla_out"):
    (CASE / sub).mkdir(parents=True, exist_ok=True)

SRC_PDB = RFA_ROOT / "test/rf2/inputs_for_test/ab_proteinmpnn_output.pdb"
INPUT_PDB = CASE / "inputs_rfa" / SRC_PDB.name
if not INPUT_PDB.exists():
    shutil.copy(SRC_PDB, INPUT_PDB)
    print(f"copied -> {INPUT_PDB}")
else:
    print(f"[skip] already staged: {INPUT_PDB}")

# Extract H/L/T per-chain FASTAs for vanilla's ColabFold input
import biotite.structure.io.pdb as pdbio
from biotite.structure import filter_amino_acids
from biotite.structure.info import one_letter_code

atoms = pdbio.PDBFile.read(str(INPUT_PDB)).get_structure(model=1)
atoms = atoms[filter_amino_acids(atoms)]
ca = atoms[atoms.atom_name == "CA"]
for c in "HLT":
    sub = ca[ca.chain_id == c]
    seq = "".join(one_letter_code(r) for r in sub.res_name).replace("?", "X")
    fasta = CASE / "a3m_vanilla" / f"chain_{c}.fasta"
    fasta.write_text(f">case_{c}\n{seq}\n")
    print(f"chain {c}: {len(sub)} residues -> {fasta.name}")

### 3.2 — Fetch per-chain ColabFold MSAs (vanilla only)

RFantibody doesn't use MSAs, so this is vanilla-side only. Each chain takes ~1-2 min via the ColabFold public API. The cell skips chains whose `case_{H,L,T}_uniref.a3m` already exists, so re-running is cheap.

In [ ]:
for c in "HLT":
    msa = CASE / "a3m_vanilla" / f"case_{c}_uniref.a3m"
    if msa.exists():
        print(f"[skip] chain {c} MSA already present ({msa.stat().st_size // 1024} KB)")
        continue
    seq = (CASE / "a3m_vanilla" / f"chain_{c}.fasta").read_text().splitlines()[1]
    print(f"fetching chain {c} ({len(seq)} aa)...")
    r = subprocess.run(
        ["uv", "run", "python", str(RFA_ROOT / "vanilla_rf2_helper.py"),
         "msa", f"case_{c}", seq, str(CASE / "a3m_vanilla")],
        cwd=str(RFA_ROOT), capture_output=True, text=True,
    )
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"MSA fetch failed for chain {c}: {r.stderr}")

### 3.3 — Run RFantibody RF2 (RF2_ab)

Uses the `rf2` CLI with the stock test config (3 recycles, deterministic seed).

In [ ]:
rfa_out = list((CASE / "rfa_out").glob("*_best.pdb"))
if rfa_out:
    print(f"[skip] RFa prediction already present: {rfa_out[0].name}")
else:
    env = os.environ.copy()
    env.update({
        "PYTHONHASHSEED": "0",
        "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
        "TORCH_DETERMINISTIC": "1",
        "TORCH_USE_CUDA_DSA": "0",
    })
    r = subprocess.run(
        ["uv", "run", "rf2",
         "--input-dir", str(CASE / "inputs_rfa"),
         "--output-dir", str(CASE / "rfa_out"),
         "--num-recycles", "3",
         "--no-cautious",
         "--seed", "42",
         "--extra", "inference.hotspot_show_proportion=0",
         "--extra", f"hydra.run.dir={CASE}/rfa_hydra"],
        cwd=str(RFA_ROOT), env=env, capture_output=True, text=True,
    )
    # Print only the cycle lines + errors to keep output scannable
    for line in r.stdout.splitlines():
        if any(t in line for t in ("Cycle", "Best pLDDT", "Processing", "Completed", "ERROR")):
            print(line)
    if r.returncode != 0:
        print("STDERR tail:", r.stderr[-1000:])
        raise RuntimeError("RFa run failed")

### 3.4 — Run vanilla RF2 (multimer)

`predict.py` with three a3m files = one multimer prediction. `-symm C1` is the default.

In [ ]:
vanilla_pred = CASE / "vanilla_out" / "case_00_pred.pdb"
if vanilla_pred.exists():
    print(f"[skip] vanilla prediction already present: {vanilla_pred.name}")
else:
    venv_py = subprocess.check_output(
        ["uv", "run", "python", "-c", "import sys; print(sys.prefix)"],
        cwd=str(RFA_ROOT), text=True,
    ).strip() + "/bin/python"
    r = subprocess.run(
        [venv_py, "predict.py",
         "-inputs",
           str(CASE / "a3m_vanilla" / "case_H_uniref.a3m"),
           str(CASE / "a3m_vanilla" / "case_L_uniref.a3m"),
           str(CASE / "a3m_vanilla" / "case_T_uniref.a3m"),
         "-prefix", str(CASE / "vanilla_out" / "case"),
         "-n_recycles", "3"],
        cwd=str(RF2_VANILLA_DIR / "network"),
        capture_output=True, text=True,
    )
    for line in r.stdout.splitlines():
        if any(t in line for t in ("Running on", "N=", "recycle", "runtime=")):
            print(line)
    if r.returncode != 0:
        print("STDERR tail:", r.stderr[-1000:])
        raise RuntimeError("vanilla RF2 run failed")

### 3.5 — Compare

The comparison script computes Kabsch Cα RMSD at three scopes (global / antibody / target / interface-only) and extracts per-chain pLDDT stats. Reference = the design input PDB that RFa was trained to re-fold.

In [ ]:
rfa_pred = next((CASE / "rfa_out").glob("*_best.pdb"))
r = subprocess.run(
    ["uv", "run", "python", "scripts/compare_rf2_vs_rfab.py",
     "--case", "case1_ab_proteinmpnn",
       str(rfa_pred), str(vanilla_pred), str(INPUT_PDB),
     "--out", str(CASE / "comparison_results.json")],
    cwd=str(RFA_ROOT), capture_output=True, text=True,
)
# Drop the biotite 'elements were guessed' warning noise
print("\n".join(l for l in r.stdout.splitlines() if l.strip()))
if r.returncode != 0:
    print("STDERR:", r.stderr)

### 3.6 — Per-residue pLDDT plot (vanilla)

The picture that makes the interpretation unambiguous: vanilla is confident on the antibody, flat-lining on the target.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

npz = np.load(CASE / "vanilla_out" / "case_00.npz")
plddt = npz["lddt"].astype(float)   # shape (L,), 0-1 scale

# Chain layout: H (0..120), L (121..226), T (227..477)
n_H, n_L, n_T = 121, 106, 251
boundaries = [n_H, n_H + n_L]
region_labels = [("H", 0, n_H), ("L", n_H, n_H + n_L), ("T", n_H + n_L, n_H + n_L + n_T)]

fig, ax = plt.subplots(figsize=(10, 3.5))
colors = {"H": "tab:blue", "L": "tab:green", "T": "tab:red"}
for name, start, end in region_labels:
    ax.plot(range(start, end), plddt[start:end], lw=1.0, color=colors[name], label=f"{name} (n={end-start})")
    ax.fill_between(range(start, end), 0, plddt[start:end], alpha=0.15, color=colors[name])
for b in boundaries:
    ax.axvline(b, color="k", lw=0.5, alpha=0.3)
ax.axhline(0.7, color="gray", lw=0.5, ls="--", alpha=0.6)
ax.text(n_H + n_L + n_T - 5, 0.72, "pLDDT = 0.7", ha="right", color="gray", fontsize=8)
ax.set_xlabel("residue index (H · L · T)")
ax.set_ylabel("vanilla RF2 pLDDT")
ax.set_ylim(0, 1)
ax.set_xlim(0, len(plddt))
ax.legend(loc="lower left", ncol=3, fontsize=9)
ax.set_title("Vanilla RF2 per-residue pLDDT — antibody chains confident, target collapsed")
plt.tight_layout()
plt.show()

print("\nVanilla per-region mean pLDDT:")
for name, start, end in region_labels:
    vals = plddt[start:end]
    p10, p50, p90 = np.percentile(vals, [10, 50, 90])
    print(f"  {name}: mean={vals.mean():.3f}  p10/p50/p90 = {p10:.2f} / {p50:.2f} / {p90:.2f}")

### Interpretation

Three things to note from the table + plot:

1. **Vanilla RF2 folds the antibody Fv correctly.** Both H and L chains sit above pLDDT 0.7 for most residues; Cα RMSD vs the reference is ~1.4 Å (RFa gets 1.16 Å — essentially tied). Antibody framework conservation gives vanilla enough MSA signal even for designed CDRs.

2. **Vanilla RF2 loses on the target chain.** pLDDT median ~0.18, RMSD ~17 Å vs reference. The designed target has ~500 UniRef30 hits vs ~10,000 for each antibody chain (MSA depth is inverted from what you'd expect on a designed complex), and vanilla RF2 relies entirely on MSA coevolution to fold it.

3. **Vanilla RF2 can't place the complex.** Both global RMSDs sit at ~18 Å. Without the hotspot channel (Case 1) and the +200 chain-offset prior (Case 2), vanilla has no signal telling it where on T the antibody should bind.

The gap is **concentrated**, not diffuse — exactly where Cases 1 and 2 predicted it would be.

---

## Gotchas (what couldn't be reproduced and why)

Two natural follow-up test cases were attempted during development of this tutorial and don't work off-the-shelf — documenting here so the next person doesn't lose the same time:

### Would-be Case 3b: real antibody + real antigen (PDB 1N8Z, trastuzumab + HER2)

1. **Too big.** Full 1N8Z is ~1015 residues — OOMs on 16 GB VRAM via vanilla RF2 multimer.
2. **Needs `REMARK PDBinfo-LABEL` CDR annotations.** Even when trimmed to Fv + HER2 domain IV (~408 residues), RFantibody's `pose_util.pose_from_remarked` rejects the PDB. Transplanting CDR labels from RFantibody's included `hu-4D5-8_Fv.pdb` is off-by-N because 1N8Z's crystal H3 is 4 residues longer than the example's. Needs an ANARCI-based CDR annotator to fix properly.

### Would-be Case 3c: Fv-only prediction (no target)

`scripts/examples/example_inputs/hu-4D5-8_Fv.pdb` is already HLT-remarked and has no T chain. Running RFa on it crashes in `RF2_util.center_and_realign_missing` because `make_t2d` masks out everything when `antibody_length == total_length`:

```
IndexError: argmin(): Expected reduction dim 1 to have non-zero size.
```

RFantibody's inference preprocessor hard-assumes a target chain is present. Vanilla RF2 handles both cases fine.

Both failures are informative — they show *where* RFantibody's preprocessing is tightly scoped to the "Ab + T design" case it was trained for.

---

## Summary

| case | what it shows | key cells |
|---|---|---|
| **1** | RF2_ab has **one** extra input scalar (hotspot), used 4× across 3 template-encoder layers | 1.1 – 1.3 |
| **2** | Pipeline Ab-specificity = four masking/offset choices, no new algorithms | 2.1 – 2.4 |
| **3** | Vanilla RF2 folds Ab Fv correctly, fails the target and the pose | 3.1 – 3.6 |

All three cases point to the same conclusion: **RF2_ab ≠ vanilla RF2 + hotspot**, but it's close. The weights can't be swapped (template widths + v2 `bind_pred` head), but the delta is small, localized, and well-understood — which is precisely what makes the failure modes in Case 3 predictable rather than mysterious.